# Day 54：微调效果评测、LoRA 权重合并与模型量化（GPTQ / AWQ / GGUF）

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第五阶段（Stage 5）——**Day 54**！今天我们将动手验证三个生产级核心能力：
> 1. **纯 Python LoRA 权重合并**（$W' = W + \frac{\alpha}{r} B @ A$），消除推理期旁路延迟；
> 2. **微调前后对比评测**（`evaluate_before_after`），量化对比 Base 与 Fine-tuned 模型的准确率及结构化格式合规率；
> 3. **大模型量化体积估算**（`estimate_quantized_size_gb`），精准对比 `FP16`、`GPTQ-4bit`、`AWQ-4bit`、`GGUF-Q4_K_M` 与 `GGUF-Q8_0`。

## 1. 环境初始化与导入 `project_iterations/day_54` 模块

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day54_dir = repo_root / "project_iterations" / "day_54"
if str(day54_dir) not in sys.path:
    sys.path.insert(0, str(day54_dir))

from zhishu_agent import (
    build_default_eval_cases,
    compare_quantization_formats,
    estimate_quantized_size_gb,
    evaluate_before_after,
    merge_lora_weights,
)

print("已加载模块路径:", day54_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_54


## 2. 纯 Python 验证 LoRA 权重合并：$W' = W + \frac{\alpha}{r} B @ A$

In [2]:
base_w = [
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
]
lora_b = [
    [1.0, 0.0],
    [0.0, 2.0],
]  # shape (2, 2)
lora_a = [
    [0.5, -0.5, 1.0],
    [0.25, 0.5, -0.25],
]  # shape (2, 3)

merged_w = merge_lora_weights(base_w, lora_a, lora_b, lora_alpha=8, lora_rank=2)
print("原始基座权重 W :", base_w)
print("合并后权重 W'  :", merged_w)
assert merged_w == [[3.0, 0.0, 7.0], [6.0, 9.0, 4.0]]
print("LoRA 权重合并校验通过！")

原始基座权重 W : [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]
合并后权重 W'  : [[3.0, 0.0, 7.0], [6.0, 9.0, 4.0]]
LoRA 权重合并校验通过！


## 3. 微调前后对比评测（准确率 & 结构化格式合规率）

In [3]:
cases = build_default_eval_cases()
report = evaluate_before_after(cases)

print(f"评测样本数: {report['total_cases']}")
print(f"准确率变化: {report['base_accuracy']:.1%} -> {report['finetuned_accuracy']:.1%} (+{report['accuracy_delta']:.1%})")
print(f"格式合规率: {report['base_format_compliance']:.1%} -> {report['finetuned_format_compliance']:.1%} (+{report['format_compliance_delta']:.1%})")
print("逐条明细:")
for item in report["case_details"]:
    print(" ", item)

assert report["improved"] is True
assert report["finetuned_accuracy"] == 1.0
assert report["finetuned_format_compliance"] == 1.0

评测样本数: 5
准确率变化: 40.0% -> 100.0% (+60.0%)
格式合规率: 20.0% -> 100.0% (+80.0%)
逐条明细:
  {'case_id': 'eval_01_tool_call_leave', 'expected_format': 'tool_call', 'base_accurate': True, 'finetuned_accurate': True, 'base_format_compliant': False, 'finetuned_format_compliant': True}
  {'case_id': 'eval_02_json_expense', 'expected_format': 'json', 'base_accurate': False, 'finetuned_accurate': True, 'base_format_compliant': False, 'finetuned_format_compliant': True}
  {'case_id': 'eval_03_sql_readonly', 'expected_format': 'sql', 'base_accurate': False, 'finetuned_accurate': True, 'base_format_compliant': False, 'finetuned_format_compliant': True}
  {'case_id': 'eval_04_tool_call_meeting', 'expected_format': 'tool_call', 'base_accurate': True, 'finetuned_accurate': True, 'base_format_compliant': True, 'finetuned_format_compliant': True}
  {'case_id': 'eval_05_json_risk_route', 'expected_format': 'json', 'base_accurate': False, 'finetuned_accurate': True, 'base_format_compliant': False, 'finetuned_form

## 4. 模型量化体积估算：FP16 vs GPTQ / AWQ / GGUF

In [4]:
for params_b in [7.0, 14.0]:
    print(f"\n=== {params_b:.0f}B 参数模型量化体积对比 ===")
    for row in compare_quantization_formats(params_b):
        print(
            f"{row['quant_format']:12s} | {row['bits_per_weight']:5.2f} bpw | "
            f"体积={row['size_gb']:6.3f} GB | 压缩比={row['compression_vs_fp16']:4.2f}x | "
            f"建议显存={row['recommended_vram_gb']:5.2f} GB"
        )

assert estimate_quantized_size_gb(7.0, "GPTQ-4bit") < estimate_quantized_size_gb(7.0, "GGUF-Q4_K_M")
assert estimate_quantized_size_gb(7.0, "GGUF-Q4_K_M") < estimate_quantized_size_gb(7.0, "Q8_0")
assert estimate_quantized_size_gb(7.0, "Q8_0") < estimate_quantized_size_gb(7.0, "FP16")
print("\n量化体积与显存估算全部校验通过！")


=== 7B 参数模型量化体积对比 ===
FP16         | 16.00 bpw | 体积=14.000 GB | 压缩比=1.00x | 建议显存=16.80 GB
GPTQ-4bit    |  4.25 bpw | 体积= 3.719 GB | 压缩比=3.76x | 建议显存= 4.46 GB
AWQ-4bit     |  4.25 bpw | 体积= 3.719 GB | 压缩比=3.76x | 建议显存= 4.46 GB
GGUF-Q4_K_M  |  4.85 bpw | 体积= 4.242 GB | 压缩比=3.30x | 建议显存= 5.09 GB
GGUF-Q8_0    |  8.50 bpw | 体积= 7.438 GB | 压缩比=1.88x | 建议显存= 8.93 GB

=== 14B 参数模型量化体积对比 ===
FP16         | 16.00 bpw | 体积=28.000 GB | 压缩比=1.00x | 建议显存=33.60 GB
GPTQ-4bit    |  4.25 bpw | 体积= 7.438 GB | 压缩比=3.76x | 建议显存= 8.93 GB
AWQ-4bit     |  4.25 bpw | 体积= 7.438 GB | 压缩比=3.76x | 建议显存= 8.93 GB
GGUF-Q4_K_M  |  4.85 bpw | 体积= 8.484 GB | 压缩比=3.30x | 建议显存=10.18 GB
GGUF-Q8_0    |  8.50 bpw | 体积=14.875 GB | 压缩比=1.88x | 建议显存=17.85 GB

量化体积与显存估算全部校验通过！
